<a href="https://colab.research.google.com/github/Ahmettufan3310/74-musteri-segmentasyonu-rfm/blob/main/notebooks/02_rfm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ucimlrepo -q
from ucimlrepo import fetch_ucirepo

df = fetch_ucirepo(id=352).data.original

temiz = df.dropna(subset=['CustomerID'])
temiz = temiz[~temiz['InvoiceNo'].astype(str).str.startswith('C')]
temiz = temiz[(temiz['Quantity'] > 0) & (temiz['UnitPrice'] > 0)]

# ürün dışı kodlar
cikar = ['POST', 'M', 'C2', 'DOT', 'BANK CHARGES', 'PADS']
temiz = temiz[~temiz['StockCode'].astype(str).isin(cikar)]

# aynı gün iptal edilen dev siparişlerin asıl satırları
hatali = [('581483', '23843'), ('541431', '23166'), ('540815', '21108'), ('556444', '22502')]
for fatura, urun in hatali:
    temiz = temiz[~((temiz['InvoiceNo'].astype(str) == fatura) & (temiz['StockCode'].astype(str) == urun))]

temiz = temiz.drop_duplicates()
print(len(temiz), temiz['CustomerID'].nunique())

391146 4333


In [ ]:
import pandas as pd

temiz['InvoiceDate'] = pd.to_datetime(temiz['InvoiceDate'])
referans = temiz['InvoiceDate'].max() + pd.Timedelta(days=1)
temiz['Tutar'] = temiz['Quantity'] * temiz['UnitPrice']

# müşteri bazında R, F, M
rfm = temiz.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda x: (referans - x.max()).days),
    Frequency=('InvoiceNo', 'nunique'),
    Monetary=('Tutar', 'sum'),
    IlkAlisveris=('InvoiceDate', lambda x: (referans - x.min()).days)
).reset_index()

print(rfm.shape)
print(rfm.head())
print(rfm.describe().round(1))

(4333, 5)
   CustomerID  Recency  Frequency  Monetary  IlkAlisveris
0     12347.0        2          7   4310.00           367
1     12348.0       75          4   1437.24           358
2     12349.0       19          1   1457.55            19
3     12350.0      310          1    294.40           310
4     12352.0       36          7   1385.74           297
       CustomerID  Recency  Frequency  Monetary  IlkAlisveris
count      4333.0   4333.0     4333.0    4333.0        4333.0
mean      15299.9     92.7        4.2    1949.2         223.3
std        1721.6    100.1        7.6    8434.5         117.9
min       12347.0      1.0        1.0       2.9           1.0
25%       13813.0     18.0        1.0     304.0         113.0
50%       15298.0     51.0        2.0     661.3         249.0
75%       16779.0    143.0        5.0    1626.0         327.0
max       18287.0    374.0      206.0  279138.0         374.0


In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler

# log dönüşümü
rfm_log = rfm.copy()
for s in ['Recency', 'Frequency', 'Monetary']:
    rfm_log[s] = np.log1p(rfm_log[s])

# ölçekleme
olcekleyici = StandardScaler()
rfm_olcekli = rfm_log.copy()
rfm_olcekli[['Recency','Frequency','Monetary']] = olcekleyici.fit_transform(rfm_log[['Recency','Frequency','Monetary']])

print(rfm_olcekli.head())
print(rfm_olcekli[['Recency','Frequency','Monetary']].describe().round(2))

   CustomerID   Recency  Frequency  Monetary  IlkAlisveris
0     12347.0 -2.040681   1.082182  1.436972           367
1     12348.0  0.371732   0.392807  0.559582           358
2     12349.0 -0.624695  -0.951156  0.570790            19
3     12350.0  1.423436  -0.951156 -0.705526           310
4     12352.0 -0.165529   1.082182  0.530437           297
       Recency  Frequency  Monetary
count  4333.00    4333.00   4333.00
mean      0.00      -0.00      0.00
std       1.00       1.00      1.00
min      -2.34      -0.95     -4.16
25%      -0.66      -0.95     -0.68
50%       0.09      -0.36     -0.06
75%       0.85       0.66      0.66
max       1.56       5.85      4.77


In [ ]:
# bir müşteriyi elle kontrol etme
mid = rfm['CustomerID'].iloc[0]
ornek = temiz[temiz['CustomerID'] == mid]
print("Müşteri:", mid)
print("Elle -> Frequency:", ornek['InvoiceNo'].nunique(),
      "| Monetary:", round((ornek['Quantity'] * ornek['UnitPrice']).sum(), 2))
print("Tablodan:")
print(rfm[rfm['CustomerID'] == mid][['Frequency', 'Monetary']])

Müşteri: 12347.0
Elle -> Frequency: 7 | Monetary: 4310.0
Tablodan:
   Frequency  Monetary
0          7    4310.0


In [ ]:
rfm.to_csv('rfm_ozgun.csv', index=False)
rfm_olcekli.to_csv('rfm_olcekli.csv', index=False)
print("Kaydedildi")

Kaydedildi
